In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q09/rewritten/o4_mini_high_q09/checkpoints/post_cell_2_try_1.pickle

trying: ['DATA_ROOT']
me:  0
trying: ['load_lineitem']
me:  1
trying: ['lineitem']


me:  1
trying: ['load_orders']
me:  2
trying: ['STORAGE_OPTS']
me:  0
trying: ['orders']


me:  2
trying: ['part']
me:  3
trying: ['load_part']
me:  3
trying: ['tpch_parent']
me:  0
trying: ['pd']
me:  0


Declaring variable DATA_ROOT
Declaring variable STORAGE_OPTS
Declaring variable tpch_parent
Declaring variable pd
Declaring variable load_lineitem
Declaring variable lineitem
Declaring variable load_orders
Declaring variable orders
Declaring variable part
Declaring variable load_part


In [4]:
%%RecordEvent
%%time
### cell 3 ###

def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = f"{data_folder}/nation.tbl"
    # explicitly define column names and dtypes to skip expensive type inference
    cols = ["N_NATIONKEY", "N_NAME", "N_REGIONKEY", "N_COMMENT"]
    dtypes = {
        "N_NATIONKEY": "int32",
        "N_NAME": "object",
        "N_REGIONKEY": "int32",
        "N_COMMENT": "object",
    }
    # read only the first four fields, avoid parsing the trailing delimiter
    df = pd.read_csv(
        data_path,
        sep="|",
        header=None,
        names=cols,
        usecols=[0, 1, 2, 3],
        dtype=dtypes,
    )
    return df

nation = load_nation(DATA_ROOT, **STORAGE_OPTS)

CPU times: user 13.9 ms, sys: 56.8 ms, total: 70.7 ms
Wall time: 74.7 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q09/rewritten/o4_mini_high_q09/checkpoints/post_cell_3_try_0.pickle

migration speed (bps): 788844070.3236037
---------------------------
variables to migrate:
lineitem 48
load_orders 144
DATA_ROOT 80
orders 48
load_part 144
load_nation 144
pd 72
tpch_parent 54
STORAGE_OPTS 64
nation 48
load_lineitem 144
part 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q09/rewritten/o4_mini_high_q09/checkpoints/post_cell_3_try_0.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables ['lineitem']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['orders']
Future variables []
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['part']
Future variables []
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables ['nation']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q09/opt_cell_exec_info_3_try_0.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[3], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q09/annotated/checkpoints/post_cell_3.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
